# 02. YOLO baseline, validation and first submission
HW2: Multiple Object Video Detection (UCU AI 2026)

Needs a GPU and Internet (for `pip` and the pretrained weights). Inputs: the competition data and the dataset saved from notebook 01.

The validation protocol is **fixed here and reused for every model**: COCO evaluation (`pycocotools`) on the val subset, ignored regions are skipped through `iscrowd=1`, up to 500 detections per frame. Reported: mAP, AP50, AP_small and inference speed. Numbers printed by Ultralytics itself are only used to pick the best epoch (they count detections inside ignored regions as errors).

## 0. Install and config

In [ ]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path
import csv, json, time

import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO

ROOT = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
SUB = Path("/kaggle/input/hw2-sub/hw2_sub")   # <-- SET ME: dataset saved from notebook 01 (copy the path from the Input panel)
WORK = Path("/kaggle/working")

MODEL = "yolo11s.pt"   # pretrained weights; try "yolov8s.pt" if this name is not available in your ultralytics version
EPOCHS = 15
IMGSZ = 640
BATCH = 16
SEED = 42
RUN_NAME = "yolo_baseline"
MAX_DET = 500          # dense scenes: the default 300 would cut detections

## 1. Dataset config
The input folder is read-only, so the yaml is written to the working folder and points to the input data. Labels are found automatically: Ultralytics replaces `/images/` by `/labels/` in every image path.

In [ ]:
names = [c["name"] for c in sorted(json.loads((SUB / "val_sub.json").read_text())["categories"], key=lambda c: c["id"])]
yaml_path = WORK / "data.yaml"
yaml_path.write_text(
    f"path: {SUB}\ntrain: train/images\nval: val/images\nnames:\n"
    + "\n".join(f"  {i}: {n}" for i, n in enumerate(names)) + "\n")
print(yaml_path.read_text())
print(json.loads((SUB / "subset_info.json").read_text()))

## 2. Train
Short run on purpose: the first goal is a working pipeline and a valid submission. Test with `EPOCHS = 1` first so that a bug does not burn the weekly GPU quota.

In [ ]:
model = YOLO(MODEL)
model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
            device=0, workers=2, project=str(WORK / "runs"), name=RUN_NAME, plots=True)

## 3. Evaluation protocol (COCO metrics on the val subset)
The same functions will be reused for all detectors, so the numbers are comparable.

In [ ]:
VAL_JSON = SUB / "val_sub.json"

def detect(model, paths, imgsz, conf=0.001, max_det=MAX_DET, batch=16):
    """Run the detector, yield (xyxy [N,4], score [N], class [N], (h, w), speed_ms) per image, in input order."""
    for res in model.predict(paths, imgsz=imgsz, conf=conf, max_det=max_det, batch=batch, stream=True, verbose=False):
        b = res.boxes
        yield (b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int),
               res.orig_shape, sum(res.speed.values()))

def _ap(E, iou=None, area=0):
    """Mean precision from the accumulated COCOeval; area index: 0 all, 1 small, 2 medium, 3 large."""
    p = E.eval["precision"]                      # [IoU, recall, class, area, maxDets]
    if iou is not None:
        p = p[np.isclose(E.params.iouThrs, iou)]
    p = p[:, :, :, area, -1]
    p = p[p > -1]
    return float(p.mean()) if p.size else float("nan")

def coco_eval(model, imgsz, conf=0.001):
    gt = COCO(str(VAL_JSON))
    ids = sorted(gt.imgs)
    paths = [str(SUB / "val" / gt.imgs[i]["file_name"]) for i in ids]
    dets, speeds = [], []
    for iid, (xyxy, sc, cl, shp, ms) in zip(ids, detect(model, paths, imgsz, conf)):
        speeds.append(ms)
        for (x1, y1, x2, y2), s, c in zip(xyxy, sc, cl):
            dets.append({"image_id": iid, "category_id": int(c) + 1, "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(s)})
    E = COCOeval(gt, gt.loadRes(dets), "bbox")
    E.params.imgIds = ids
    E.params.maxDets = [1, 10, MAX_DET]
    E.evaluate(); E.accumulate()
    return {"mAP": _ap(E), "AP50": _ap(E, 0.5), "AP75": _ap(E, 0.75),
            "AP_small": _ap(E, area=1), "AP_medium": _ap(E, area=2), "AP_large": _ap(E, area=3),
            "ms_per_img": float(np.mean(speeds)), "n_dets": len(dets)}

In [ ]:
best = sorted((WORK / "runs").rglob("best.pt"), key=lambda p: p.stat().st_mtime)[-1]
print("weights:", best)
model = YOLO(str(best))
res = coco_eval(model, IMGSZ)
print({k: round(v, 4) for k, v in res.items()})

# row for experiments_log.md
print(f"| 1 | YOLO baseline, {MODEL}, imgsz {IMGSZ}, {EPOCHS} ep | first run | {res['mAP']:.3f} | {res['AP50']:.3f} | {res['AP_small']:.3f} | {res['ms_per_img']:.1f} | | |")

## 4. Submission
**Check the format first** on the competition page (tab *Evaluation*) and adjust only the next cell: column names and order, `x y w h` vs `x1 y1 x2 y2`, class id from 0 or 1, what `image_id` is.
The frames to predict come from `test_images.csv` (column `image_id` is used as given), so every listed frame gets its predictions.

In [ ]:
# --- ADJUST TO THE COMPETITION FORMAT (tab Evaluation) ---
HEADER = ["image_id", "category_id", "x", "y", "w", "h", "score"]
CLASS_ID_OFFSET = 1          # model classes are 0..7, COCO ids are 1..8

def make_row(image_id, cls, score, x1, y1, x2, y2):
    return [image_id, int(cls) + CLASS_ID_OFFSET,
            round(float(x1), 2), round(float(y1), 2), round(float(x2 - x1), 2), round(float(y2 - y1), 2),
            round(float(score), 5)]
# -----------------------------------------------------------

In [ ]:
test_df = pd.read_csv(ROOT / "test_images.csv")
paths = [str(ROOT / f) for f in test_df.file_name]
print(len(paths), "test frames | first:", paths[0])

size_mismatch, n_rows, t0 = 0, 0, time.time()
with open(WORK / "submission.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(HEADER)
    for r, (xyxy, sc, cl, (h, w), ms) in zip(test_df.itertuples(), detect(model, paths, IMGSZ)):
        size_mismatch += (w, h) != (r.width, r.height)
        for (x1, y1, x2, y2), s, c in zip(xyxy, sc, cl):
            wr.writerow(make_row(r.image_id, c, s, x1, y1, x2, y2)); n_rows += 1
print(f"{n_rows} rows in {time.time() - t0:.0f}s | frames whose real size differs from the CSV: {size_mismatch}")
print(pd.read_csv(WORK / "submission.csv").head())

## Next
1. If `size_mismatch` is not 0, stop and check which coordinate frame the competition expects (CSV size or real image size).
2. *Save Version* (Save & Run All), then submit `submission.csv` from the Output tab. The score must be above the baseline on the leaderboard.
3. Put the results (val metrics and public score) into `experiments_log.md`, download both notebooks and commit them.